<a href="https://colab.research.google.com/github/davejeon/dsr_agentic_workflow/blob/main/Exercise_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
# Install packages
%pip install langchain-google-genai
%pip install urllib3

In [22]:
# Import packages
from langchain.messages import AIMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import os
from google.colab import userdata
from langchain.tools import tool


In [23]:
# Connect notebook to langsmith
os.environ['LANGSMITH_TRACING'] = userdata.get('LANGSMITH_TRACING')
os.environ['LANGSMITH_ENDPOINT'] = userdata.get('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = userdata.get('LANGSMITH_PROJECT')
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
# Test the output
userdata.get('LANGSMITH_PROJECT')

'dsr_agentic_ai'

- Simplest way to create a tool is with the @tool decorator.

- The function's docstring becomes the tool's description that helps the model understand when to use it.

In [31]:
# Creating a basic tool
# Modify the answer by removing the caps lock text.
@tool("calculator")
def multiplication_calculator(expression:str)->str:
  """Evaluate a mathematical expression and return the result.

  Args:
    expresion: A mathematical expression like:
       - "y by z", or
       - "y*z", or
       - "y multiplied by z", or
       - "y times z", or
       - "y x z"

    In which case, y and z are both numbers, whether floats or integers.

    Before entering the expression into the function, make sure that this rewritten, if necessary, into the format y*z.

    RETURN THE ANSWER AS A SINGLE VALUE WITH NO ADDITIONAL TEXT

  Returns:
    Dictionary with 'result' or 'error'
  """
  try:
    # Evaluate the expression
    result = eval(expression)
    return {
        "success'": True,
        "result": result,
        "expression": expression
    }
  except Exception as e:
    return {
        "success": False,
        "error": str(e),
        "expression":expression
    }

In [32]:
ai_context = """You are a helpful assistant"""

human_message = """What is 60 times 24?"""


In [33]:
agent = create_agent(model = "google_genai:gemini-3.1-flash-lite", tools=[multiplication_calculator])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

In [34]:
print(result)

{'messages': [HumanMessage(content='What is 60 times 24?', additional_kwargs={}, response_metadata={}, id='b6bb75ef-88d4-43aa-ba31-adf615e6da38'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'calculator', 'arguments': '{"expression": "60*24"}'}, '__gemini_function_call_thought_signatures__': {'c58e506c-fc41-4c4d-a5be-6e3c98cac280': 'EjQKMgEMOdbHDXE57BAHU7tC/eG6oh21HQaF+eZq/D5eBmA4YmaxTJeUwxVY3PCOiw0jH9hC'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e2732-bb27-79a1-a45a-88a5e180a0ac-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '60*24'}, 'id': 'c58e506c-fc41-4c4d-a5be-6e3c98cac280', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 189, 'output_tokens': 18, 'total_tokens': 207, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='{"success\'": true, "result": 1440, "expression": "60*24"}', name='c

In [35]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
output_list

[[],
 [{'type': 'text',
   'text': '1440',
   'extras': {'signature': 'EjQKMgEMOdbHUSywIF/5E6fCJRbne/TP1wmY2CuYDbXEAxluN5PZFBypAkEhWG8NmqKKC0CC'}}]]

In [36]:
# If you want a neater output
print(output_list[1][0]["text"])

1440


In [54]:
# Let's try something more interesting
# Let's try something more interesting
from bs4 import BeautifulSoup
import re
import urllib3

@tool("weather page")
def get_weather(expression: str) -> str:
  """Gets the current weather for Berlin by scraping weather data from HTML response.

  Args:
    expression: The user's weather query (e.g., "What's the weather in Berlin?")

  Returns:
    A string containing the parsed weather information or an error message
  """

  try:
      # Create a PoolManager instance for sending requests
      http = urllib3.PoolManager()
      url = 'https://www.theweathernetwork.com/en/city/de/berlin/berlin/current'

      # Send GET request
      resp = http.request("GET", url)

      # Parse HTML with BeautifulSoup
      soup = BeautifulSoup(resp.data, 'html.parser')

      # Extract weather information - adjust selectors based on actual page structure
      # Common patterns for weather sites:

      # Try to find temperature
      temp_element = soup.find('span', class_=re.compile(r'temp|temperature'))
      temperature = temp_element.text.strip() if temp_element else "N/A"

      # Try to find weather condition
      condition_element = soup.find('div', class_=re.compile(r'condition|weather-desc'))
      condition = condition_element.text.strip() if condition_element else "N/A"

      # Try to find location
      location_element = soup.find('h1', class_=re.compile(r'location|city'))
      location = location_element.text.strip() if location_element else "Berlin"

      # Try to find feels like temperature
      feels_like_element = soup.find('span', class_=re.compile(r'feels-like|feels'))
      feels_like = feels_like_element.text.strip() if feels_like_element else "N/A"

      # Construct a readable response
      if temperature != "N/A" or condition != "N/A":
          weather_info = f"Current weather in {location}: {temperature}, {condition}"
          if feels_like != "N/A":
              weather_info += f" (feels like {feels_like})"
          return weather_info
      else:
          # Fallback: return raw HTML snippet for debugging
          return f"Could not parse weather data. Raw response length: {len(resp.data)} characters"

  except Exception as e:
      return f"Error fetching weather: {str(e)}"

In [55]:

ai_context = """You are a helpful assistant"""

human_message = """What's the weather in Berlin like?"""

agent = create_agent(model = "google_genai:gemini-3.1-flash-lite", tools=[multiplication_calculator])
result = agent.invoke(
        {"messages": [
            {"role": "user",
             "content": f"{human_message}"}]},
          context=ai_context)

In [56]:
ai_messages = [
    m for m in result["messages"]
    if isinstance(m, AIMessage)
]
#print(ai_messages)
output_list=[message.content for message in ai_messages]
output_list

[[],
 [{'type': 'text',
   'text': 'As of today, November 21, 2024, the weather in Berlin is quite cold and wintery.\n\nIt is currently hovering around **0°C to 2°C (32°F to 36°F)**, with overcast skies and a high chance of light snow showers or sleet throughout the day. It is quite breezy, making it feel colder than the actual temperature.\n\nIf you are heading out, definitely dress warmly with a heavy coat, scarf, and gloves!',
   'extras': {'signature': 'EjQKMgEMOdbHFknK0pxI4IK7sGr6mQe/uzoy2LIJHsLAz6FCH+wwACwMgTyIBt/xOB16mobA'}}]]